___

# <font color= #EF4444> **Speech Radicalization Analysis** </font>
#### <font color= #f0565e> `Deep Learning`</font>
<Strong> Sofía Maldonado, Oscar Josué Rocha & Viviana Toledo </Strong>

_12/05/2026._

___

In [1]:
# Imports
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from datasets import Dataset
import json
import pandas as pd

/home/sofi/Documents/vscode/speech_radicalization_analysis/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


To create the embeddings we're going to use BERTweet, a custom version of BERT specifically for use with tweets, which is what our data consists of.

In [2]:
MODEL_NAME = "vinai/bertweet-base"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 38534.93it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/bertweet-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(64001, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(130, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropou

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else 'cpu'
print(DEVICE)
model.to(DEVICE)

cuda


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(64001, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(130, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropou

In [5]:
df = pd.read_csv('../data/processed/speech_classified.csv')
df = df[['clean_text', 'annotation_label']]
df

,clean_text,annotation_label
0,"china, this is what you are known for: animalr...",Hate Speech
1,catabuserschina visa workers and their employe...,Hate Speech
2,china madeinchina chinatravel china doesn't ha...,Extreme Speech
3,"they are ugly psychopaths, monsters!!!",Extreme Speech
4,and its culture of cruelty and brutality,Hate Speech
...,...,...
6213,its time for our weekly economic update on isr...,Hate Speech
6214,any doubt....,Hate Speech
6215,not this republican. neveragain allow another ...,Hate Speech
6216,"that's one way to filter your feed, i guess. p...",Enraged Speech


In [6]:
tweets = df['clean_text'].tolist()
labels = df['annotation_label'].tolist()

In [7]:
def get_embeddings(tweets, batch_size=32):
    all_embeddings = []
    for i in range(0, len(tweets), batch_size):
        batch = tweets[i: i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        ).to(DEVICE)

        with torch.no_grad():
            output = model(**encoded)
        
        hidden_states = output.last_hidden_state

        embeddings = hidden_states[:, 0, :]

        all_embeddings.append(embeddings.cpu().numpy())
        print(f'Processed {batch_size} tweets')
    
    return np.vstack(all_embeddings)


In [8]:
embeddings = get_embeddings(tweets, 32)
print(f"Shape: {embeddings.shape}")

Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets
Processed 32 tweets


In [10]:
print(embeddings)

[[ 0.00270845  0.21146582 -0.01707458 ... -0.12540321 -0.07531948
  -0.19043514]
 [ 0.09999809  0.32456154  0.11904883 ... -0.1952425  -0.22930032
  -0.16286427]
 [ 0.1363184   0.21985567  0.07602783 ... -0.01111341 -0.01267161
  -0.02720094]
 ...
 [-0.12309562  0.15676436  0.01672103 ... -0.10203682 -0.09813331
  -0.02040674]
 [ 0.15909247  0.32616678  0.09166586 ...  0.0495679   0.25605983
   0.04516199]
 [-0.12316097  0.15612204  0.12046503 ... -0.06900726 -0.2565016
   0.00373487]]


In [17]:
hf_dataset = Dataset.from_dict({
    'embeddings': embeddings.tolist(),
    'labels': labels
})

hf_dataset.save_to_disk('../data/processed/bertweet_embeddings')

Saving the dataset (1/1 shards): 100%|██████████| 6218/6218 [00:00<00:00, 266977.00 examples/s]
